In [1]:
import os
os.environ["OMP_NUM_THREADS"] = "1"   # keeps CPU load low for long runs


In [2]:
# ================================================
# Cell 1: Overlay load and IP discovery
# ================================================
from pynq import Overlay, allocate
import numpy as np, time, math, os
np.set_printoptions(linewidth=120, edgeitems=4, suppress=True)

# ---- All files are in the same folder as this notebook ----
BIT_PATH = "/home/xilinx/conv_opt.bit"
HWH_PATH = "/home/xilinx/conv_opt.hwh"

# Binary parameter files
BIN_INPUT   = "./conv_input.bin"
BIN_WEIGHTS = "./conv_weights.bin"
BIN_BIASES  = "./conv_biases.bin"

# ---- AlexNet Conv1 parameters ----
M, I = 3, 224       # input depth, height/width
F, K = 64, 11       # number of filters, kernel size
S, P = 4, 2         # stride, padding
O = 55              # output height/width
N = F               # output depth

# Load overlay
ol = Overlay(BIT_PATH)
ol.download()
print("Loaded overlay. IPs found:", list(ol.ip_dict.keys()))

# Locate convolution IP automatically
ip_name = None
for k in ol.ip_dict.keys():
    if "conv" in k.lower():
        ip_name = k
        break
if ip_name is None:
    for k in ol.ip_dict.keys():
        if "processing_system" not in k.lower():
            ip_name = k
            break

ip = getattr(ol, ip_name)
print("Using IP:", ip_name)
print(ip.register_map)

# Helper: 64-bit pointer writer
def write_ptr64(ip, base_name, addr):
    low  = int(addr & 0xFFFFFFFF)
    high = int((addr >> 32) & 0xFFFFFFFF)
    setattr(ip.register_map, base_name + "_1", low)
    setattr(ip.register_map, base_name + "_2", high)

# Identify pointer fields
rm = ip.register_map
ptr_fields = [f for f in dir(rm) if f.endswith("_1") or f.endswith("_2")]
bases = sorted(set(f[:-2] for f in ptr_fields if f.endswith("_1")))
print("Pointer bases found:", bases)

# Guess register mapping
map_guess = {"input": None, "weights": None, "biases": None, "output": None}
for b in bases:
    bl = b.lower()
    if map_guess["input"]   is None and "in" in bl: map_guess["input"] = b
    if map_guess["weights"] is None and "weight" in bl: map_guess["weights"] = b
    if map_guess["biases"]  is None and "bias" in bl: map_guess["biases"] = b
    if map_guess["output"]  is None and "out" in bl: map_guess["output"] = b

# Fill in remaining if needed
unassigned = [b for b in bases if b not in map_guess.values()]
for key in ["input","weights","biases","output"]:
    if map_guess[key] is None and unassigned:
        map_guess[key] = unassigned.pop(0)

print("Register mapping guess:", map_guess)


Loaded overlay. IPs found: ['conv_0/s_axi_control', 'conv_0/s_axi_control_r', 'processing_system7_0']
Using IP: conv_0/s_axi_control
RegisterMap {
  CTRL = Register(AP_START=0, AP_DONE=0, AP_IDLE=1, AP_READY=0, RESERVED_1=0, AUTO_RESTART=0, RESERVED_2=0, INTERRUPT=0, RESERVED_3=0),
  GIER = Register(Enable=0, RESERVED=0),
  IP_IER = Register(CHAN0_INT_EN=0, CHAN1_INT_EN=0, RESERVED_0=0),
  IP_ISR = Register(CHAN0_INT_ST=0, CHAN1_INT_ST=0, RESERVED_0=0),
  input_fm_0_1 = Register(input_fm_0=write-only),
  input_fm_0_2 = Register(input_fm_0=write-only),
  input_fm_1_1 = Register(input_fm_1=write-only),
  input_fm_1_2 = Register(input_fm_1=write-only),
  input_fm_2_1 = Register(input_fm_2=write-only),
  input_fm_2_2 = Register(input_fm_2=write-only),
  weights_0_1 = Register(weights_0=write-only),
  weights_0_2 = Register(weights_0=write-only),
  weights_1_1 = Register(weights_1=write-only),
  weights_1_2 = Register(weights_1=write-only),
  weights_2_1 = Register(weights_2=write-only),
  

In [3]:
# ================================================
# Cell 2: Buffers & binary loads
# ================================================
input_fm  = allocate((M, I, I),    dtype=np.float32)
weights   = allocate((F, M, K, K), dtype=np.float32)
biases    = allocate((F,),         dtype=np.float32)
output_fm = allocate((N, O, O),    dtype=np.float32)

# Load input, weights, biases
input_fm[:]  = np.fromfile(BIN_INPUT,   dtype=np.float32, count=M*I*I).reshape(M, I, I)
weights[:]   = np.fromfile(BIN_WEIGHTS, dtype=np.float32, count=F*M*K*K).reshape(F, M, K, K)
biases[:]    = np.fromfile(BIN_BIASES,  dtype=np.float32, count=F)
output_fm.fill(0)

print("Loaded input_fm, weights, biases.")
print(" input_fm :", input_fm.shape)
print(" weights  :", weights.shape)
print(" biases   :", biases.shape)
print(" output_fm:", output_fm.shape)


Loaded input_fm, weights, biases.
 input_fm : (3, 224, 224)
 weights  : (64, 3, 11, 11)
 biases   : (64,)
 output_fm: (64, 55, 55)


In [4]:
# ===============================================================
# Cell: CPU Reference Convolution (for batch-safe execution)
# ===============================================================
import numpy as np

def conv2d_cpu_ref(input_fm, weights, biases, K, S, P=2):
    """
    CPU reference implementation of the 2D convolution layer
    (matches FPGA output dimensions, used for batch execution)
    """
    M, I, _ = input_fm.shape
    F, _, _, _ = weights.shape

    # Apply padding (2 pixels)
    padded = np.pad(input_fm, ((0, 0), (P, P), (P, P)), mode='constant')

    # Compute output size
    O = ((I - K + 2 * P) // S) + 1
    output = np.zeros((F, O, O), dtype=np.float32)

    # Convolution
    for f in range(F):
        for r in range(O):
            for c in range(O):
                in_r0 = r * S
                in_c0 = c * S
                acc = biases[f]
                for m in range(M):
                    acc += np.sum(
                        weights[f, m, :, :] *
                        padded[m, in_r0:in_r0 + K, in_c0:in_c0 + K]
                    )
                output[f, r, c] = max(acc, 0.0)  # ReLU
    return output


In [5]:
# ===============================================================
# Safe full-size CPU convolution in small batches (prevents crash)
# ===============================================================
import time, sys

print("Running CPU convolution in safe batches... please wait.", flush=True)

F_total = weights.shape[0]   # total filters (e.g. 64)
batch_size = 8               # process 8 filters at a time (fits in memory)

M, I, _ = input_fm.shape
K, S, P = 11, 4, 2
O = ((I - K + 2 * P) // S) + 1
C_cpu = np.zeros((F_total, O, O), dtype=np.float32)

t0 = time.perf_counter()

# Run 8 filters at a time to reduce memory & thread load
for f0 in range(0, F_total, batch_size):
    f1 = min(f0 + batch_size, F_total)
    sub_weights = weights[f0:f1]
    sub_biases  = biases[f0:f1]
    C_cpu[f0:f1] = conv2d_cpu_ref(input_fm, sub_weights, sub_biases, K, S, P)
    print(f"  Batch {f0}–{f1} done.", flush=True)

t1 = time.perf_counter()
cpu_ms = (t1 - t0) * 1e3
print(f"CPU conv total time: {cpu_ms:.2f} ms")
print(f"C_cpu shape: {C_cpu.shape}")


Running CPU convolution in safe batches... please wait.
  Batch 0–8 done.
  Batch 8–16 done.
  Batch 16–24 done.
  Batch 24–32 done.
  Batch 32–40 done.
  Batch 40–48 done.
  Batch 48–56 done.
  Batch 56–64 done.
CPU conv total time: 381801.66 ms
C_cpu shape: (64, 55, 55)


In [6]:
# ================================================
# Cell 4: FPGA run (kernel-only timing)
# ================================================
write_ptr64(ip, map_guess["input"],   input_fm.physical_address)
write_ptr64(ip, map_guess["weights"], weights.physical_address)
write_ptr64(ip, map_guess["biases"],  biases.physical_address)
write_ptr64(ip, map_guess["output"],  output_fm.physical_address)

# Warm-up
ip.write(0x00, 1)
while (ip.read(0x00) & 0x2) == 0:
    pass

fpga_times = []
for _ in range(3):
    output_fm.fill(0)
    t0 = time.perf_counter()
    ip.write(0x00, 1)
    while (ip.read(0x00) & 0x2) == 0:
        pass
    t1 = time.perf_counter()
    fpga_times.append((t1 - t0) * 1e3)

fpga_ms = float(np.median(fpga_times))
print("FPGA run times (ms):", [f"{x:.3f}" for x in fpga_times])
print(f"FPGA median: {fpga_ms:.3f} ms")


FPGA run times (ms): ['6848.290', '6848.268', '6848.276']
FPGA median: 6848.276 ms


In [7]:
# ================================================
# Cell 5: Correctness & performance (vs CPU only)
# ================================================
C_cpu = conv2d_cpu_ref(input_fm, weights, biases, K, S)
C_fpga = output_fm

# Compute Mean Squared Error
mse_hw_vs_cpu = np.mean((C_fpga - C_cpu)**2)

# Performance metrics
macs = F * O * O * (M * K * K)
ops  = 2.0 * macs
cpu_gops  = (ops / (cpu_ms/1e3)) / 1e9
fpga_gops = (ops / (fpga_ms/1e3)) / 1e9
speedup = cpu_ms / fpga_ms if fpga_ms > 0 else float("inf")

print("=== Correctness ===")
print(f"MSE (HW vs CPU): {mse_hw_vs_cpu:.6e}")
print("PASS" if mse_hw_vs_cpu < 1e-4 else "WARNING: output mismatch")

print("\\n=== Performance ===")
print(f"CPU  time : {cpu_ms:.3f} ms")
print(f"FPGA time : {fpga_ms:.3f} ms")
print(f"CPU  perf : {cpu_gops:.3f} GFLOP/s")
print(f"FPGA perf : {fpga_gops:.3f} GFLOP/s")
print(f"Speedup    : {speedup:.2f}x")


=== Correctness ===
MSE (HW vs CPU): 1.241778e+00
\n=== Performance ===
CPU  time : 381801.658 ms
FPGA time : 6848.276 ms
CPU  perf : 0.000 GFLOP/s
FPGA perf : 0.021 GFLOP/s
Speedup    : 55.75x


In [8]:
# ================================================
# Cell 6: Quick output inspection
# ================================================
def peek_tensor(name, X, c=8):
    print(f"\\n{name} shape={X.shape}")
    if X.ndim == 3:
        print(X[0, :c, :c])
    elif X.ndim == 2:
        print(X[:c, :c])
    else:
        print(X[:c])

peek_tensor("input_fm", input_fm)
peek_tensor("weights[0]", weights[0])
peek_tensor("output_fpga", output_fm)
peek_tensor("cpu_output", C_cpu)


\ninput_fm shape=(3, 224, 224)
[[ 0.          0.          0.          0.          0.          0.          0.          0.        ]
 [ 0.          0.          0.         -0.8335474  -0.88492167 -0.8677969  -0.67942464 -0.59380084]
 [-0.5424266  -0.42255333 -0.35405433  0.          0.         -0.79929787 -0.79929787 -0.78217316]
 [-0.4054286  -0.49105233 -0.55955136 -0.49105233 -0.37117907  0.          0.         -0.8335474 ]
 [-0.42255333 -0.43967807 -0.38830382 -0.45680285 -0.5424266  -0.5424266  -0.42255333  0.        ]
 [-0.4054286  -0.4054286  -0.42255333 -0.45680285 -0.4739276  -0.5424266  -0.5424266  -0.5253019 ]
 [-0.38830382 -0.38830382 -0.42255333 -0.45680285 -0.45680285 -0.38830382 -0.4739276  -0.59380084]
 [-0.35405433 -0.38830382 -0.38830382 -0.38830382 -0.45680285 -0.4739276  -0.43967807 -0.42255333]]
\nweights[0] shape=(3, 11, 11)
[[ 0.11863963  0.09406868  0.09543519  0.10515065  0.10291012  0.0672244   0.05046584  0.05012476]
 [ 0.07488242  0.03894044  0.05297883  0.07595